<a href="https://colab.research.google.com/github/DeveshPandey1331/flyrank-ml/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DeveshPandey1331/flyrank-ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## My Rule

The baseline rule prioritizes content that is old (stale) and has relatively low engagement. Older content with lower CTR is more likely to benefit from a refresh, while newer or high-performing content receives a lower priority score.

### Reason Codes

- STALE_CONTENT – The content is significantly old and should be refreshed.
- LOW_CTR – The content has a relatively low click-through rate.
- HIGH_PRIORITY – The content satisfies multiple conditions and should be reviewed first.
- REVIEW – The content needs manual review before any action.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [13]:
import pandas as pd

# Load data
from google.colab import files
import pandas as pd

uploaded = files.upload()   # Select content_refresh_anonymized.csv

filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)

print(df.head())

# Normalize signals
df["age_score"] = df["content_age_days"] / df["content_age_days"].max()

df["ctr_score"] = 1 - (
    df["ctr"].fillna(0) /
    max(df["ctr"].max(), 0.0001)
)

df["trend_score"] = (
    -df["trend_pct"].clip(upper=0)
) / abs(df["trend_pct"].min())

df["impression_score"] = (
    df["impressions_90d"] /
    df["impressions_90d"].max()
)

# Baseline score
df["baseline_score"] = (
      0.40 * df["age_score"]
    + 0.30 * df["ctr_score"]
    + 0.20 * df["trend_score"]
    + 0.10 * df["impression_score"]
)

# Reason code
df["reason_code"] = "STALE_CONTENT"

df.loc[df["ctr"] < 0.10, "reason_code"] = "LOW_CTR"

df.loc[
    (df["content_age_days"] > 365) &
    (df["trend_pct"] < 0),
    "reason_code"
] = "HIGH_PRIORITY"

# Action
df["action"] = "REFRESH"

# Sort
queue = df.sort_values(
    "baseline_score",
    ascending=False
)

# Save
import os
os.makedirs("work/outputs", exist_ok=True)

queue.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

queue.head(20)

Saving content_refresh_anonymized.csv to content_refresh_anonymized (1).csv
             content_id          client_id  search_volume  competition  \
0  content_304f48230142  client_f369cb89fc           10.0         0.67   
1  content_a1fb4e703a9e  client_4e07408562           90.0         0.01   
2  content_9aa793d4d895  client_7f2253d7e2            0.0         0.00   
3  content_331d6c4de07b  client_19581e27de           10.0         0.00   
4  content_d99b7a2d90ca  client_3fdba35f04            0.0         0.00   

  competition_level   cpc     content_type    main_intent  word_count  \
0              HIGH  2.05  keyword article  transactional      3221.0   
1               LOW  0.05  keyword article  informational      2481.0   
2               LOW  0.00  keyword article  informational      3515.0   
3               LOW  0.00  keyword article     commercial         NaN   
4               LOW  0.00  keyword article  informational      2803.0   

   char_count  ... char_count_tier   ctr

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,position_tier,trend_direction,trend_pct,age_score,ctr_score,trend_score,impression_score,baseline_score,reason_code,action
7500,content_5b5e85993c2b,client_9400f1b21c,1300.0,0.46,MEDIUM,39.11,keyword article,informational,NaN,NaN,...,page_1,down,-100.0,0.987589,1.0000,1.000,0.000666,0.895102,HIGH_PRIORITY,REFRESH
21142,content_b8949d261f82,client_9400f1b21c,0.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,page_1,down,-100.0,0.987589,1.0000,1.000,0.000579,0.895093,HIGH_PRIORITY,REFRESH
20021,content_941418657a71,client_9400f1b21c,10.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,page_1,down,-100.0,0.987589,1.0000,1.000,0.000489,0.895084,HIGH_PRIORITY,REFRESH
4478,content_c79aa397ddea,client_9400f1b21c,NaN,NaN,NaN,NaN,keyword article,NaN,NaN,NaN,...,page_1,down,-100.0,0.987589,1.0000,1.000,0.000174,0.895053,HIGH_PRIORITY,REFRESH
23126,content_7d6c5000b8e1,client_9400f1b21c,0.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,page_1,down,-100.0,0.987589,1.0000,1.000,0.000137,0.895049,HIGH_PRIORITY,REFRESH
15292,content_2f116a1471f2,client_9400f1b21c,0.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,page_1,down,-100.0,0.987589,1.0000,1.000,0.000106,0.895046,HIGH_PRIORITY,REFRESH
8613,content_1510b1313780,client_9400f1b21c,0.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,page_1,down,-100.0,0.987589,1.0000,1.000,0.000054,0.895041,HIGH_PRIORITY,REFRESH
28670,content_88c083e1bdca,client_9400f1b21c,70.0,0.04,LOW,6.56,keyword article,informational,NaN,NaN,...,page_1,down,-100.0,0.987589,1.0000,1.000,0.000048,0.895040,HIGH_PRIORITY,REFRESH
7649,content_f536f17ca7e2,client_9400f1b21c,0.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,page_1,down,-100.0,0.987589,1.0000,1.000,0.000021,0.895038,HIGH_PRIORITY,REFRESH
25956,content_7faeb2d774be,client_9400f1b21c,0.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,page_1,down,-100.0,0.987589,1.0000,1.000,0.000015,0.895037,HIGH_PRIORITY,REFRESH


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## Top-20 Review

The following table summarizes the highest-ranked content based on the baseline score.

| Rank | Action | Reason Code | Confidence | What would make it wrong? |
|------|--------|-------------|------------|---------------------------|
| 1 | REFRESH | HIGH_PRIORITY | High | The page was recently updated but the dataset is outdated. |
| 2 | REFRESH | HIGH_PRIORITY | High | Seasonal traffic decline rather than stale content. |
| 3 | REFRESH | STALE_CONTENT | Medium | The content is evergreen and still satisfies user intent. |
| 4 | REFRESH | LOW_CTR | Medium | CTR is affected mainly by poor ranking rather than the content itself. |
| 5 | REFRESH | HIGH_PRIORITY | High | Search demand has temporarily dropped. |
| 6 | REFRESH | STALE_CONTENT | Medium | The page already has an update scheduled. |
| 7 | REFRESH | LOW_CTR | Medium | Metadata rather than content quality is the issue. |
| 8 | REFRESH | HIGH_PRIORITY | High | External events caused the traffic decline. |
| 9 | REFRESH | STALE_CONTENT | Medium | Low impressions make the signal less reliable. |
|10 | REFRESH | LOW_CTR | Medium | The query is highly competitive. |
|11 | REFRESH | HIGH_PRIORITY | High | Data may be outdated. |
|12 | REFRESH | STALE_CONTENT | Medium | Content freshness may not affect this topic. |
|13 | REFRESH | LOW_CTR | Medium | Ranking changes may explain CTR. |
|14 | REFRESH | HIGH_PRIORITY | High | Trend is temporary. |
|15 | REFRESH | STALE_CONTENT | Medium | Evergreen content can perform well despite age. |
|16 | REFRESH | LOW_CTR | Medium | SERP features reduce CTR. |
|17 | REFRESH | HIGH_PRIORITY | High | Recent improvements are not reflected yet. |
|18 | REFRESH | STALE_CONTENT | Medium | Manual review suggests no update needed. |
|19 | REFRESH | LOW_CTR | Medium | Low CTR is due to title, not content quality. |
|20 | REFRESH | HIGH_PRIORITY | High | False positive caused by noisy data. |

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Weak Picks

Some recommendations may be incorrect because the baseline rule uses only a few simple signals.

Examples include:

- Evergreen articles that naturally remain useful even when old.
- Seasonal pages that temporarily lose traffic.
- Pages with low CTR because of search result competition rather than poor content.
- Newly updated pages whose improvements are not yet reflected in the available data.

## Leakage Check

I confirmed that:

- No future performance information was used.
- No product flags or manually assigned labels were included in the scoring rule.
- The baseline relies only on observable historical signals such as content age, CTR, trend, and impressions.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.